# Development 0: environment probes

Standalone. Needs no repo and no files from the laptop. Connect to a Colab
GPU runtime, then run the cells one at a time, top to bottom. Every cell is
independent, so a failure in one does not block the next.

No cell prints a token. Secrets are only reported as found or not found.

In [1]:
# --- 1. Runtime: GPU, disk, Python, NumPy, PyTorch ---
import shutil, subprocess, sys

gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True)
print("GPU          :", gpu.stdout.strip() if gpu.returncode == 0 else "NONE - check the runtime type")
total, used, free = shutil.disk_usage("/content")
print(f"Disk /content: {free / 1e9:.0f} GB free of {total / 1e9:.0f} GB")
print("Python       :", sys.version.split()[0])
for name in ("numpy", "torch", "torchvision", "pandas", "pyarrow", "einops", "safetensors", "cv2"):
    try:
        module = __import__(name)
        print(f"{name:13}:", getattr(module, "__version__", "present"))
    except ImportError:
        print(f"{name:13}: MISSING")

GPU          : NONE - check the runtime type
Disk /content: 221 GB free of 242 GB
Python       : 3.13.15
numpy        : 2.1.3
torch        : 2.11.0+cpu
torchvision  : 0.26.0+cpu
pandas       : 2.2.3
pyarrow      : 23.0.1
einops       : 0.8.2
safetensors  : 0.8.0
cv2          : 5.0.0


In [2]:
# --- 2. C++ toolchain, needed by `06_cpp_core` ---
import subprocess

for tool in ("g++", "cmake", "make", "ninja"):
    result = subprocess.run(f"{tool} --version", shell=True, capture_output=True, text=True)
    print(f"{tool:9}:", result.stdout.splitlines()[0] if result.returncode == 0 and result.stdout else "MISSING")
try:
    import pybind11
    print("pybind11 :", pybind11.__version__)
except ImportError:
    print("pybind11 : MISSING (pip-installable, not a blocker)")
eigen = subprocess.run("ls -d /usr/include/eigen3 2>/dev/null", shell=True, capture_output=True, text=True)
print("Eigen    :", "present" if eigen.stdout.strip() else "MISSING (header-only, easy to fetch)")

g++      : g++ (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
cmake    : cmake version 3.31.10
make     : GNU Make 4.3
ninja    : MISSING
pybind11 : MISSING (pip-installable, not a blocker)
Eigen    : MISSING (header-only, easy to fetch)


In [3]:
# --- 3. Are Colab secrets readable through the VS Code extension? ---
# Prints found / not found only. Never a value.
SECRET_NAMES = ("HF_TOKEN",)
SECRETS = {}
try:
    from google.colab import userdata
    print("google.colab.userdata: importable")
    for name in SECRET_NAMES:
        try:
            value = userdata.get(name)
            SECRETS[name] = value or None
            print(f"  {name:9}:", "found" if value else "empty")
        except Exception as err:
            SECRETS[name] = None
            print(f"  {name:9}: NOT readable ({type(err).__name__})")
except ImportError:
    print("google.colab.userdata: NOT importable - this kernel is not a Colab runtime")

google.colab.userdata: importable
  HF_TOKEN : NOT readable (TimeoutException)


In [4]:
# --- 4. Fallback: type any secret that cell 3 could not read (input is hidden) ---
# Skip this cell if cell 3 found everything. Leave a prompt blank to skip that secret.
from getpass import getpass

if "SECRETS" not in globals():
    SECRETS = {}
for name in ("HF_TOKEN",):
    if not SECRETS.get(name):
        typed = getpass(f"{name} (hidden, Enter to skip): ").strip()
        SECRETS[name] = typed or None
print({name: ("set" if value else "not set") for name, value in SECRETS.items()})

{'HF_TOKEN': 'set'}


In [5]:
# --- 5. Does the Hugging Face token reach the gated checkpoint? ---
import requests

token = globals().get("SECRETS", {}).get("HF_TOKEN")
if not token:
    print("no HF_TOKEN available - run cell 3 or 4 first")
else:
    auth = {"Authorization": f"Bearer {token}"}
    who = requests.get("https://huggingface.co/api/whoami-v2", headers=auth, timeout=30)
    print("token accepted :", who.ok, "| account:", who.json().get("name") if who.ok else f"status {who.status_code}")
    head = requests.head("https://huggingface.co/facebook/VGGT-Omega/resolve/main/vggt_omega_1b_512.pt",
                         headers=auth, timeout=30, allow_redirects=False)
    print("gated 512 checkpoint reachable:", head.status_code in (200, 302, 307), f"(status {head.status_code})")

token accepted : True | account: saverino
gated 512 checkpoint reachable: True (status 302)


In [6]:
# --- 6. Does Google Drive mount through the extension? Decides the persistence design. ---
# Expect a sign-in prompt. If nothing appears after a minute, interrupt the cell and report that.
import time
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    probe = Path("/content/drive/MyDrive/vggt-omega-aura-benchmark/_probe.txt")
    probe.parent.mkdir(parents=True, exist_ok=True)
    stamp = str(time.time())
    probe.write_text(stamp)
    print("Drive mounted  : True")
    print("write and read : ", probe.read_text() == stamp)
    probe.unlink()
    print("probe file removed, folder kept:", probe.parent)
except Exception as err:
    print("Drive mount FAILED:", type(err).__name__, str(err)[:300])

Mounted at /content/drive
Drive mounted  : True
write and read :  True
probe file removed, folder kept: /content/drive/MyDrive/vggt-omega-aura-benchmark


In [7]:
# --- 7. Download check from Hugging Face: one small real file from the dataset ---
import time, requests

url = "https://huggingface.co/datasets/fzi-forschungszentrum-informatik/FZI-AURA/resolve/3404bd6b8fcd6eed53a0ec7610650a6393aabb49/metadata/v1.0/chunks.parquet"
start = time.time()
response = requests.get(url, timeout=120)
elapsed = time.time() - start
print("status:", response.status_code, "| bytes:", len(response.content), f"| {elapsed:.2f} s")
print("Too small to measure bandwidth. It only confirms the dataset is reachable at the pinned revision.")

status: 200 | bytes: 222905 | 1.49 s
Too small to measure bandwidth. It only confirms the dataset is reachable at the pinned revision.


## Manual check: Server Mounting

1. In VS Code settings, search for `colab.serverMounting` and turn it on.
2. Run the command `Colab: Mount Server To Workspace`.
3. See whether `/content` appears in the Explorer panel.

## What to report back

Copy the text output of every cell, plus three answers: did the Drive
sign-in prompt appear and complete, did Server Mounting show the runtime
files, and were Colab secrets readable in cell 3 or did you need cell 4.